# Script to run GPT using the batch API

In [1]:
import json
import os

import tiktoken
from openai import OpenAI

from scripts.prompting.prompts import task_description_v2
from scripts.prompting.run_llms import open_input_file, prepare_instructions, get_input_text

In [2]:
def create_batch_request(prompt, request_id, model_id):
    return {
        "custom_id": request_id,
        "method": "POST",
        "url": "/v1/chat/completions",
        "body": {
            "model": model_id,
            "messages": [
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            "max_tokens": 4096
        }
    }


In [28]:
def from_dataset_to_batch_req(test_folder, train_folder, dot_train_data, instructions_func, output_file, num_of_files, num_of_examples, model_id):
    enc = tiktoken.encoding_for_model(model_id)
    json_lines = list()

    examples = prepare_instructions(train_folder, dot_train_data, num_of_examples)
    final_instructions = instructions_func(examples)

    total_tokens = 0
    count = 0
    for i, file1 in enumerate(os.listdir(test_folder)):
        if count == num_of_files:
            break

        count += 1
        data = open_input_file(f'{test_folder}/{file1}')
        text = get_input_text(data)
        prompt = final_instructions + '\n' + text

        # Count the number of tokens in the prompt
        encoded_text = enc.encode(prompt)
        total_tokens += len(encoded_text)
        print(f'Number of tokens in prompt={len(encoded_text)}')

        req = create_batch_request(prompt, file1, model_id)
        json_lines.append(req)
    
    print(f'Total number of tokens in all prompts={total_tokens}')
    split = 1
    if total_tokens > 90000: 
        if total_tokens/2 < 90000:
            split = 2
        elif total_tokens/3 < 90000:
            split = 3
        else: 
            split = 4
    
    chunk_size = len(json_lines) // split
    chunks = [json_lines[i:i + chunk_size] for i in range(0, len(json_lines), chunk_size)]
    for i in range(len(chunks)):
        with open(f'{output_file}_chunk{i}.jsonl', 'w', encoding='utf8') as file1:
            for req in chunks[i]:
                file1.write(json.dumps(req) + '\n')


In [4]:
def upload_batch_request(client, input_request_jsonl):
    batch_input_file = client.files.create(
        file=open(input_request_jsonl, 'rb'),
        purpose='batch'
    )

    print(f'Batch input file with id {batch_input_file.id} uploaded')
    return batch_input_file.id


In [5]:
def run_batch(client, batch_input_file_id):
    create = client.batches.create(input_file_id=batch_input_file_id, endpoint="/v1/chat/completions",
                                   completion_window="24h", metadata={"description": "eventfull gpt4o batch job"})

    print(f'Batch object created: {create}')
    return create.id


In [6]:
def check_batch_status(client, batch_id, output_file):
    batch = client.batches.retrieve(batch_id)
    print(f'Batch status: {batch}')

    if batch.status == 'completed':
        file_response = client.files.content(batch.output_file_id)
        with open(output_file, 'w', encoding='utf8') as file:
            for req in file_response.text.split('\n'):
                file.write(json.dumps(req) + '\n')
        print(file_response.text)


In [7]:
def convert_response(response_file, final_output_file):
    converted = dict()
    with (open(response_file, 'r', encoding='utf8') as file1):
        for line in file1:
            if not line.strip():
                continue

            response = json.loads(line)
            res_loads = json.loads(response)
            converted[res_loads['custom_id']] = {'target': res_loads['response']['body']['choices'][0]['message']['content']}

    with open(final_output_file, 'w', encoding='utf8') as file2:
        json.dump(converted, file2, indent=4)


Step-0: Initialize the client

In [41]:
with open('/Users/aloneirew/workspace/DenseEREGraph/data/my_data/api_key.txt', 'r') as file:
    api_key = file.readline().strip()

_client = OpenAI(api_key=api_key)

_test_folder = '/Users/aloneirew/workspace/DenseEREGraph/data/EventFullTrainExports/test'
_train_folder = '/Users/aloneirew/workspace/DenseEREGraph/data/EventFullTrainExports/dev'
_dot_train_data = open_input_file('/Users/aloneirew/workspace/DenseEREGraph/data/DOT_format/EventFull_dev_dot.json')

_output_file = '/Users/aloneirew/workspace/DenseEREGraph/data/my_data/batch_req/eventfull_request_gpt4o_2exmlps_chunk1.jsonl'
_response_file = '/Users/aloneirew/workspace/DenseEREGraph/data/my_data/batch_req/eventfull_response_gpt4o_2exmlps_merged.jsonl'
_final_output_file = '/Users/aloneirew/workspace/DenseEREGraph/data/my_data/batch_req/eventfull_gpt4o_2exmlps_task_description_v2.json'

_instructions_func = task_description_v2
_num_of_files_to_prepare = -1
_num_of_examples = 2
_model_id = "gpt-4o"

Step-1: Create the request and save it to a file

In [29]:
print("Running Step-1")
from_dataset_to_batch_req(_test_folder, _train_folder, _dot_train_data, _instructions_func, _output_file, _num_of_files_to_prepare, _num_of_examples, _model_id)

Running Step-1
The mentions are-['fire', 'fire', 'recovery', 'failed', 'used', 'filed', 'calling', 'filed', 'visited', 'trapped', 'identified', 'searching', 'recover', 'ignored', 'leased', 'lived']
The input text is-– The death toll from Friday 's horrific warehouse <fire(49)> has risen to 36 , making it California 's deadliest building <fire(33)> since the devastation of the 1906 San Francisco earthquake . As <recovery(13)> efforts at the " Ghost Ship " warehouse continue , many are wondering why inspectors <failed(24)> to act despite repeated complaints that the cluttered building was being illegally <used(20)> for living and entertainment purposes . NBC reports that at least 10 complaints were <filed(2)> over the last 18 years , including a 2007 complaint <calling(34)> the building " a nuisance or substandard or hazardous or injurious . " Two of the complaints were <filed(3)> just last month . Officials say inspectors <visited(4)> Nov. 17 , but left after they could n't get in . A r

Step-2: upload the request to the server

In [37]:
print("Running Step-2")
print(_output_file)
_input_request_jsonl = _output_file
_batch_input_file_id = upload_batch_request(_client, _input_request_jsonl)

Running Step-2
/Users/aloneirew/workspace/DenseEREGraph/data/my_data/batch_req/eventfull_request_gpt4o_2exmlps_chunk1.jsonl
Batch input file with id file-1NqZxlxIj3G2t07PfXZMZpWO uploaded


Step-3: Run the batch

In [38]:
print("Running Step-3")
print(_batch_input_file_id)
_batch_run_id = run_batch(_client, _batch_input_file_id)

Running Step-3
file-1NqZxlxIj3G2t07PfXZMZpWO
Batch object created: Batch(id='batch_fmgyPUbX0gtuFkjnBCJrxN9m', completion_window='24h', created_at=1726328736, endpoint='/v1/chat/completions', input_file_id='file-1NqZxlxIj3G2t07PfXZMZpWO', object='batch', status='validating', cancelled_at=None, cancelling_at=None, completed_at=None, error_file_id=None, errors=None, expired_at=None, expires_at=1726415136, failed_at=None, finalizing_at=None, in_progress_at=None, metadata={'description': 'eventfull gpt4o batch job'}, output_file_id=None, request_counts=BatchRequestCounts(completed=0, failed=0, total=0))


Step-4: Check the status of the batch

In [40]:
print("Running Step-4")
print(_batch_run_id)
check_batch_status(_client, _batch_run_id, _response_file)

Running Step-4
batch_fmgyPUbX0gtuFkjnBCJrxN9m
Batch status: Batch(id='batch_fmgyPUbX0gtuFkjnBCJrxN9m', completion_window='24h', created_at=1726328736, endpoint='/v1/chat/completions', input_file_id='file-1NqZxlxIj3G2t07PfXZMZpWO', object='batch', status='completed', cancelled_at=None, cancelling_at=None, completed_at=1726328979, error_file_id=None, errors=None, expired_at=None, expires_at=1726415136, failed_at=None, finalizing_at=1726328978, in_progress_at=1726328737, metadata={'description': 'eventfull gpt4o batch job'}, output_file_id='file-W4ku8NmEM2RSTToUhoTKSo3V', request_counts=BatchRequestCounts(completed=10, failed=0, total=10))
{"id": "batch_req_fdsAilHTpiSXtWap8D034Daa", "custom_id": "9_final.json", "response": {"status_code": 200, "request_id": "305022fadaeadc7bc4cb6005cace40cf", "body": {"id": "chatcmpl-A7PFvuet0q0OELyHxrnaogL9LPVnl", "object": "chat.completion", "created": 1726328743, "model": "gpt-4o-2024-05-13", "choices": [{"index": 0, "message": {"role": "assistant", "

Step-5: Convert response to DOT format response

In [42]:
print("Running Step-5")
convert_response(_response_file, _final_output_file)
print("Done!")

Running Step-5
Done!


Cancelling a batch

In [33]:
_client.batches.cancel("batch_xnWSUbQRSq3ZSZCwj67gtPLi")

ConflictError: Error code: 409 - {'error': {'message': "Cannot cancel a batch with status 'failed'.", 'type': 'invalid_request_error', 'param': None, 'code': None}}